# Task A – Benchmarking Pipeline (Automated)

This notebook mirrors the repository pipeline for Task A. It downloads the raw phishing datasets, builds the processed splits, trains the classical machine learning baselines with hyper-parameter search, optionally fine-tunes DistilBERT, and finally runs the enhanced evaluation suite that produces metrics and plots.

## 1. Environment setup

Install dependencies when running in a fresh environment (e.g. Kaggle, Colab). Skip this cell if the requirements are already installed.

In [1]:
# Uncomment when running on a clean runtime
!pip install -q -r requirements.txt
!pip install --upgrade pip
!D:\Users\alexa\anaconda3\envs\University\python.exe -m pip install --upgrade pip

## 2. Configure paths and ensure datasets

The helper below adds the project to `sys.path`, prepares the working directories, and automatically fetches the Hugging Face phishing corpora if they are missing locally. Set `CYRADAR_MAX_ROWS` to a smaller value when prototyping to subsample the larger dataset.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(f"Expected to run from the project root, got {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

RAW_DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
REPORT_DIR = PROJECT_ROOT / "reports"

for path in (RAW_DATA_DIR, PROCESSED_DIR, ARTIFACT_DIR, REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")
print(f"Artifacts directory: {ARTIFACT_DIR}")
print(f"Reports directory: {REPORT_DIR}")
print(f"CYRADAR_MAX_ROWS (optional dev helper): {os.getenv('CYRADAR_MAX_ROWS')}")

In [ ]:
from src.common.dataset_downloader import ensure_raw_datasets

print("Ensuring phishing datasets are available ...")
resolved = list(ensure_raw_datasets(RAW_DATA_DIR))
for path in resolved:
    print(f" - {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size / 1024**2:.2f} MB)")

## 3. Preprocess and create train/validation/test splits

This step mirrors `src/task_a_benchmark/preprocess.py`: it normalizes the text, removes HTML, deduplicates messages, and writes the three splits under `data/processed/`.

In [ ]:
from src.task_a_benchmark import preprocess

def describe_split(name: str, df):
    return {
        "split": name,
        "rows": len(df),
        "phishing": int((df['label'] == 1).sum()),
        "legit": int((df['label'] == 0).sum()),
    }

train_df, val_df, test_df = preprocess.preprocess_and_split()
summary = [describe_split("train", train_df), describe_split("val", val_df), describe_split("test", test_df)]
summary

In [ ]:
train_df.head()

## 4. Train tuned classical baselines

`train_models` performs grid-search tuning for the Logistic Regression, Linear SVM, Random Forest, and XGBoost pipelines, then persists the best model bundle and per-model metrics under `artifacts/` and `reports/`.

In [ ]:
from src.task_a_benchmark import train_ml
import json

ml_metrics = train_ml.train_models()
print(json.dumps(ml_metrics, indent=2))

## 5. (Optional) Fine-tune DistilBERT

Toggle `RUN_TRANSFORMER` to control whether the transformer experiment executes. A single epoch is usually sufficient for the thesis benchmarks and keeps runtime manageable.

In [ ]:
RUN_TRANSFORMER = False  # set to True to fine-tune DistilBERT

if RUN_TRANSFORMER:
    from src.task_a_benchmark import train_transformer
    transformer_metrics = train_transformer.train_transformer(num_train_epochs=1.0, batch_size=8)
    display(transformer_metrics)
else:
    print("Skipping transformer fine-tuning. Set RUN_TRANSFORMER = True to enable.")

## 6. Evaluate and generate reports

The evaluation stage loads every trained model (including the optional transformer), scores the held-out test split, computes calibration/ROC/PR curves, confusion matrices, and per-source breakdowns, and writes everything to `reports/`.

In [ ]:
from src.task_a_benchmark import evaluate
import json

metrics_summary = evaluate.evaluate_all()
print(json.dumps(metrics_summary, indent=2))

In [ ]:
from pathlib import Path
import json

best_meta_path = PROJECT_ROOT / "artifacts" / "best_model_meta.json"
if best_meta_path.exists():
    with open(best_meta_path, "r", encoding="utf-8") as fh:
        best_meta = json.load(fh)
    print("Best model metadata:")
    print(json.dumps(best_meta, indent=2))
else:
    print("best_model_meta.json not found. Ensure the training step completed successfully.")